<div style="border-top:4px solid #0f766e;padding:28px 0 18px">
<div style="color:#0f766e;font-size:13px;font-weight:700">LAB 01 · DATA WAREHOUSING WITH APACHE DORIS</div>
<h1>连接 Doris，查询第一批订单</h1>
<p>从十笔订单出发，完成连接、建表、写入和地区分析，并解释订单金额与已支付金额的区别。</p>
<p>约 25 分钟 · 合成订单数据 · 目标版本 Doris 4.1.3</p>
</div>

[讲义](course1_introduction_to_apache_doris.md) · [测验](quiz1_doris_fundamentals.ipynb) · [课程目录](../README.md)

完成后你会得到十笔订单、总订单金额 1400.00，以及 EAST/WEST 两个地区的汇总结果。
本节不需要 S3 密钥，也不做性能测试。请按顺序运行单元，快捷键为 Shift + Enter。


### 初始化实验工具

下一格只加载 Python 工具和显示样式，不连接数据库、不启动 Docker、不写入数据。
出现“实验工具已就绪”后再继续。重启内核后，需要从这里重新执行。

如果出现 ModuleNotFoundError，先按[环境说明](../../environments/single-node/README.md)
安装依赖，并确认当前 Notebook 使用的是安装这些依赖的 Python 环境。


In [ ]:
from pathlib import Path
import os
import sys

COURSE_ROOT = next(
    path for path in (Path.cwd(), *Path.cwd().parents)
    if (path / "dw_course").is_dir()
)
if str(COURSE_ROOT) not in sys.path:
    sys.path.insert(0, str(COURSE_ROOT))

from dw_course import WarehouseLab
from dw_course.runtime import expect, fixture
from dw_course.schema import ORDER_COLUMNS, order_rows
from dw_course.docker_runtime import prepare_environment
from dw_course.ui import card, install_styles

install_styles()
card("尚未连接数据库或启动容器。", "ok", "实验工具已就绪")


## 1. 选择实验环境

选择一种方式，不要同时配置两套环境：

| 方式 | 配置方法 | 会发生什么 |
| --- | --- | --- |
| 讲师已提供 Doris | 保持 USE_DOCKER 为 False，填写 FE_HOST / FE_PORT | 只连接已有实例，不管理服务器进程 |
| 自己准备 Docker 沙箱 | 把 USE_DOCKER 改为 True，先启动 Docker Desktop / Engine | 启动课程 02 独立沙箱，使用 FE 端口 52030 |

配置格优先沿用讲师在启动 Jupyter 时设置的 DW_* 参数。没有预置时，默认连接本机 9030。
**127.0.0.1 是运行 Jupyter 内核的机器，不一定是打开浏览器的 Mac。**

本 Lab 会创建指定的实验库，并在第 4 步删除、重建其中的 **d01_orders**。
确认这张表是可重置的教学数据后，把 ALLOW_LAB_WRITES 改为 True。
其他表不在本节重置范围内；不要填写生产实例或他人的实验库。

实验库名必须以 dw_course_l1_ 开头。多人共用实例时可各加自己的后缀。
密码从进程环境 DW_PASSWORD 读取；需要交互输入时取消 getpass 行的注释，不要把密码写入文件。


In [ ]:
USE_DOCKER = os.environ.get("DW_START_SANDBOX", "no") == "yes"
ALLOW_LAB_WRITES = os.environ.get("DW_ALLOW_WRITES", "no") == "yes"

FE_HOST = os.environ.get("DW_HOST", "127.0.0.1")
FE_PORT = int(os.environ.get("DW_PORT", "9030"))
DATABASE = os.environ.get("DW_DATABASE", "dw_course_l1_demo")
USER = os.environ.get("DW_USER", "root")
# from getpass import getpass
# os.environ["DW_PASSWORD"] = getpass("Doris password: ")

print("环境方式：", "课程 Docker 沙箱" if USE_DOCKER else "已有 Doris")
print("实验数据库：", DATABASE)
print("允许重建本节表：", ALLOW_LAB_WRITES)


### 启动（可选）并连接

检查上一格输出，再运行下一格。没有确认写入时会停止，不会启动容器。

Docker 模式会校验课程 Compose 文件、启动沙箱、等待健康检查通过并测试 SELECT 1，
然后配置连接地址。首次拉取镜像可能需要数分钟。
已有实例模式不会启动或重启 Doris。

WarehouseLab 会连接 FE，创建并选择刚才指定的实验数据库，
设置本会话时区为 +08:00。本节先关闭会话 Group Commit，便于逐次观察写入结果。


In [ ]:
if not ALLOW_LAB_WRITES:
    raise RuntimeError("请先确认 d01_orders 可以重建，再将 ALLOW_LAB_WRITES 设为 True。")

from dw_course.runtime import identifier
identifier(DATABASE)
if not DATABASE.startswith("dw_course_l1_"):
    raise ValueError("请使用 dw_course_l1_ 开头的独立实验库。")

os.environ.update({
    "DW_HOST": FE_HOST,
    "DW_PORT": str(FE_PORT),
    "DW_DATABASE": DATABASE,
    "DW_USER": USER,
    "DW_ALLOW_WRITES": "yes",
    "DW_START_SANDBOX": "yes" if USE_DOCKER else "no",
})
if USE_DOCKER:
    prepare_environment()

lab = WarehouseLab()
lab.sql("SELECT 1 AS connection_ok, DATABASE() AS current_database", title="连接与当前数据库");


**预期结果：** connection_ok 为 1，current_database 等于你指定的实验库。

**遇到问题：**

- Connection refused：核对 FE 查询端口及服务是否已启动，不要误用 FE HTTP 端口。
- Access denied：确认账号、密码以及创建实验库和表所需权限。
- Docker 启动失败：确认 Docker 正在运行、镜像可下载且端口没有冲突。详见[环境排查](../../environments/single-node/README.md)。
- 不要通过停止他人的服务、删除数据库或 Docker 数据卷来解决连接失败。


## 2. 检查 FE 与 BE

FE 接收 SQL 并规划查询；BE 执行查询，在本节的存算一体环境中也保存内部表数据。
先确认节点存活，再开始写入。


In [ ]:
lab.sql("SELECT VERSION() AS protocol_version, @@version_comment AS version_comment", title="连接版本信息");
lab.sql("SHOW FRONTENDS", title="FE 节点");
lab.sql("SHOW BACKENDS", title="BE 节点");


**预期结果：** 课程单节点沙箱有一个存活 FE 和一个存活 BE，Alive 为 true。
讲师提供的集群可能有更多节点。记录 FE/BE 的 Version 字段；如果是开发构建，
不要把它当成正式 4.1.3。SELECT VERSION() 可能返回协议兼容版本。

如果 BE 不存活，先处理环境问题，不要继续建表、写入来测试是否“碰巧能成功”。


## 3. 理解订单样本

仓库提供十笔合成订单，每行是一笔订单的初始快照。

| 字段 | 含义 |
| --- | --- |
| order_id、customer_id | 订单号、客户号 |
| order_amount | 订单金额，使用 DECIMAL(12,2) 保留两位小数 |
| status | 初始状态 CREATED，表示已创建，尚未支付 |
| paid_amount、refund_amount | 已支付与已退款金额，初始均为 0.00 |
| region | 地区 EAST 或 WEST |
| event_version、event_id、event_time | 初始版本 1、事件标识、事件时间，后续用于变更实验 |

先预测一下：SUM(order_amount) 与 SUM(paid_amount) 会相同吗？
本样本订单金额合计 1400.00，但尚未产生支付，所以已支付金额合计应为 0.00。


## 4. 创建第一张内部表

内部表的数据由 Doris 管理。下面直接展示完整 DDL，不用辅助函数隐藏建表语句。

- BIGINT 保存标识；DECIMAL 保存定点金额；VARCHAR 保存状态和地区。
- DUPLICATE KEY(order_id) 指定排序键，**不会强制订单号唯一**。
- HASH(order_id) BUCKETS 1 让本节小数据放在一个桶中。
- replication_num=1 适用于本节单 BE 实验，不能代表生产容错配置。

**重置提示：** 下一格会删除当前实验库中原有的 d01_orders。
如果你在这张表里做了需要保留的练习，先导出结果。它不会删除整个数据库。


In [ ]:
lab.execute("DROP TABLE IF EXISTS d01_orders")
lab.execute("""
CREATE TABLE d01_orders (
    order_id BIGINT NOT NULL,
    customer_id BIGINT NOT NULL,
    order_amount DECIMAL(12,2) NOT NULL,
    status VARCHAR(20) NOT NULL,
    event_version BIGINT NOT NULL,
    event_id VARCHAR(32) NOT NULL,
    event_time DATETIME NOT NULL,
    paid_amount DECIMAL(12,2) NOT NULL,
    refund_amount DECIMAL(12,2) NOT NULL,
    region VARCHAR(16) NOT NULL
)
DUPLICATE KEY(order_id)
DISTRIBUTED BY HASH(order_id) BUCKETS 1
PROPERTIES ("replication_num" = "1")
""")
lab.sql("DESC d01_orders", title="订单表字段");
lab.sql("SHOW CREATE TABLE d01_orders", title="实际建表定义");


**预期结果：** 字段列表包含上面十个字段，建表定义保留 Duplicate Key、
按 order_id 哈希分桶和单副本设置。Doris 可能把属性规范化显示为等价形式。

建表成功只说明结构已经存在，不代表数据已进入表。下一步才真正写入订单。


## 5. 写入十笔订单

INSERT 后显式列出字段名，可以看清每个值写入哪一列。
本节用一条 SQL 写入十行，便于理解；D05 再学习文件批量导入。

**不要单独反复运行本格。** Duplicate Key 表不会去重。
完整重跑时，先执行第 4 步重建，再执行本步，之后运行检查。


In [ ]:
lab.execute("""
INSERT INTO d01_orders (
    order_id, customer_id, order_amount, status, event_version,
    event_id, event_time, paid_amount, refund_amount, region
) VALUES
(1001, 101, 100.00, 'CREATED', 1, 'S1001', '2026-01-01 09:00:00', 0.00, 0.00, 'EAST'),
(1002, 102, 200.00, 'CREATED', 1, 'S1002', '2026-01-01 09:00:00', 0.00, 0.00, 'WEST'),
(1003, 103, 150.00, 'CREATED', 1, 'S1003', '2026-01-01 09:00:00', 0.00, 0.00, 'EAST'),
(1004, 104, 80.00, 'CREATED', 1, 'S1004', '2026-01-01 09:00:00', 0.00, 0.00, 'WEST'),
(1005, 105, 120.00, 'CREATED', 1, 'S1005', '2026-01-01 09:00:00', 0.00, 0.00, 'EAST'),
(1006, 106, 60.00, 'CREATED', 1, 'S1006', '2026-01-01 09:00:00', 0.00, 0.00, 'WEST'),
(1007, 107, 300.00, 'CREATED', 1, 'S1007', '2026-01-01 09:00:00', 0.00, 0.00, 'EAST'),
(1008, 108, 90.00, 'CREATED', 1, 'S1008', '2026-01-01 09:00:00', 0.00, 0.00, 'WEST'),
(1009, 109, 50.00, 'CREATED', 1, 'S1009', '2026-01-01 09:00:00', 0.00, 0.00, 'EAST'),
(1010, 110, 250.00, 'CREATED', 1, 'S1010', '2026-01-01 09:00:00', 0.00, 0.00, 'WEST')
""")
lab.sql("""
SELECT order_id, status, order_amount, paid_amount, region
FROM d01_orders
ORDER BY order_id
""", title="核对十笔订单");


**预期结果：** 按 order_id 顺序显示 1001–1010，共十行，status 均为 CREATED。
例如订单 1001 的金额为 100.00，已支付为 0.00，地区为 EAST。

如果看到二十行，通常是重复执行了 INSERT。先查看明细确认原因，
再完整重跑第 4、5 步；不要改预期行数来让检查通过。


## 6. 核对总量，也核对明细

COUNT(*) 统计记录行数；SUM 汇总对应金额列。因为当前样本是一单一行，
本次行数才等于订单数。若将来存入多条状态记录，这个等式就不一定成立。

除聚合值外，下面还将所有字段逐行与仓库的基准样本比较，
避免“总金额碰巧相同，明细却写错了”。校验工具只负责比较，不生成你的 SQL。


In [ ]:
lab.sql("""
SELECT COUNT(*) AS row_count,
       SUM(order_amount) AS order_amount,
       SUM(paid_amount) AS paid_amount,
       SUM(refund_amount) AS refund_amount
FROM d01_orders
""", title="订单总量与金额")

expect(
    lab.query("SELECT COUNT(*), SUM(order_amount), SUM(paid_amount), SUM(refund_amount) FROM d01_orders"),
    [(10, "1400.00", "0.00", "0.00")],
)

actual = lab.sql("""
SELECT order_id, customer_id, order_amount, status, event_version,
       event_id, CAST(event_time AS STRING) AS event_time,
       paid_amount, refund_amount, region
FROM d01_orders
ORDER BY order_id
""", title="完整明细核对")
expect(list(actual.itertuples(index=False, name=None)), order_rows(fixture("orders.json")))


**预期结果：** 十行、订单金额 1400.00、已支付与已退款均为 0.00；
聚合与明细检查都通过。

思考：这是否说明已经收到 1400.00 的货款？不是。
此处所有订单尚未支付，不能把订单金额当作实收收入。


## 7. 回答业务问题：各地区的订单情况

GROUP BY 将相同地区的记录归为一组。对每一组，分别计算行数、订单金额和已支付金额。
ORDER BY 让结果展示顺序固定，便于比较与检查。


In [ ]:
regional_sql = """
SELECT region,
       COUNT(*) AS order_count,
       SUM(order_amount) AS order_amount,
       SUM(paid_amount) AS paid_amount
FROM d01_orders
GROUP BY region
ORDER BY region
"""
lab.sql(regional_sql, title="地区订单分析")
expect(lab.query(regional_sql), [
    ("EAST", 5, "720.00", "0.00"),
    ("WEST", 5, "680.00", "0.00"),
])


**预期结果：**

| region | order_count | order_amount | paid_amount |
| --- | ---: | ---: | ---: |
| EAST | 5 | 720.00 | 0.00 |
| WEST | 5 | 680.00 | 0.00 |

EAST 的订单金额更高，两地的订单数相同。还不能由这组结果推断利润、支付转化率
或未来销量，因为样本没有提供这些结论所需的信息。


## 8. 自己动手：找出金额至少 150.00 的订单

尝试写一条只读 SELECT，返回 order_id、region、order_amount，按订单号排序。
可以在下方参考代码前插入一个代码单元，调用 lab.sql 展示你的 SQL。

**先预测结果再查询：** 应有四笔订单，金额合计 900.00。
注意“至少”包括等于 150.00 的订单。下面给出参考答案，建议先自己完成。


In [ ]:
lab.sql("""
SELECT order_id, region, order_amount
FROM d01_orders
WHERE order_amount >= 150.00
ORDER BY order_id
""", title="金额筛选：参考答案")

expect(
    lab.query("SELECT order_id FROM d01_orders WHERE order_amount >= 150.00 ORDER BY order_id"),
    [(1002,), (1003,), (1007,), (1010,)],
)
lab.sql("""
SELECT COUNT(*) AS order_count, SUM(order_amount) AS order_amount
FROM d01_orders
WHERE order_amount >= 150.00
""", title="筛选后的订单数与金额")
expect(
    lab.query("SELECT COUNT(*), SUM(order_amount) FROM d01_orders WHERE order_amount >= 150.00"),
    [(4, "900.00")],
)


## Lab 完成

你已经完成连接、检查节点、显式建表、写入十笔订单、核对明细、按地区汇总和金额筛选。
完成前，试着用自己的话回答：

1. 为什么连接 FE，但数据存储和查询执行还需要 BE？
2. 为什么订单金额 1400.00 不代表已经收到同样金额的货款？
3. 为什么不能单独重复运行这张表的 INSERT？
4. 为什么既要看总量，也要检查明细？

继续 [Quiz 1](quiz1_doris_fundamentals.ipynb)，然后进入
[D02：观察存储与写入批次](../module02-architecture/course2_doris_architecture.md)。

### 重跑与结束

- 重跑本节：从初始化开始顺序执行，第 4 步会重建本节自己的表。
- 当前 Notebook 保持连接，方便你继续写只读查询。完成后关闭内核即可释放连接。
- 如需停止自己启动的 Docker 沙箱，按[环境说明](../../environments/single-node/README.md)
  中的课程专用 stop 命令操作；不要删除数据卷。后续继续学习时保留沙箱运行即可。
- 本实验不验证实时接入、生产性能或多副本恢复。
